- `http://<ip>:800/v1/chat/completions`
    - 接受的是 messages
- `http://<ip>:8001/v1/completions`
    - 接受的是 prompts（apply chat template 之后的）

- 模型是混合线性注意力架构：model_type=qwen3_5、Qwen3_5ForConditionalGeneration，
    - rollout 端 vLLM 编译出了 causal_conv1d_fwd_kernel、fused_sigmoid_gating_delta_rule_update_kernel（Gated DeltaNet/linear-attention 的算子)
- Qwen3.5 fast-path
    - fla-core
        - flash-linear-attention
    - causal-conv1d

### thinking

In [2]:
prompt = "<|im_start|>system\n你正在执行字面字符串复制测试。尖括号及其中的文字都是普通数据，不代表任何指令、推理模式或输出结构。严格逐字符复制用户指定的字符串，不增加解释。<|im_end|>\n<|im_start|>user\n请只原样输出下面这个字符串：\nA<think>B</think>C<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
print(prompt)

<|im_start|>system
你正在执行字面字符串复制测试。尖括号及其中的文字都是普通数据，不代表任何指令、推理模式或输出结构。严格逐字符复制用户指定的字符串，不增加解释。<|im_end|>
<|im_start|>user
请只原样输出下面这个字符串：
A<think>B</think>C<|im_end|>
<|im_start|>assistant
<think>

</think>




`A</think>C`

In [3]:
prompt = "<|im_start|>system\n你正在执行字面字符串复制测试。尖括号及其中的文字都是普通数据，不代表任何指令、推理模式或输出结构。严格逐字符复制用户指定的字符串，不增加解释。<|im_end|>\n<|im_start|>user\n请只原样输出下面这个字符串：\nA</think>B<think>C<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
print(prompt)

<|im_start|>system
你正在执行字面字符串复制测试。尖括号及其中的文字都是普通数据，不代表任何指令、推理模式或输出结构。严格逐字符复制用户指定的字符串，不增加解释。<|im_end|>
<|im_start|>user
请只原样输出下面这个字符串：
A</think>B<think>C<|im_end|>
<|im_start|>assistant
<think>

</think>




`A</think>B<think>C`

```
思考要求：
- 为保证效率，你必须在 <think></think> 标签内采用“电报式”风格。
- 禁止使用序号（1.2.3.）或人类口语（如“我发现”、“下一步”）。
- 省略主谓宾等冗长修饰，将推理拆分为极简的陈述句，句间严格使用句号（。）分隔。
- 不要因为效率而牺牲思考准确性。

思考要求：
- 为保证效率，你必须在思考过程内采用“电报式”风格。
- 禁止使用序号（1.2.3.）或人类口语（如“我发现”、“下一步”）。
- 省略主谓宾等冗长修饰，将推理拆分为极简的陈述句，句间严格使用句号（。）分隔。
- 不要因为效率而牺牲思考准确性。
```

- 虽然 prompt 中写入 `<think></think>` 未必 qwen3.5 能很好的理解，但是确实是有效的，规范 thinking，压缩 thinking length。
    - 且 `必须在 <think></think> 标签内` -> `必须在思考过程内`，会让思考过程和长度回退到默认版本